# REWIND search: type a sentence, get clips back, and score it against the ground truth

Runs **after `pipeline.ipynb`** (it needs the clip captions and the SigLIP hand-off in `outputs/batch/`). Writes
`outputs/batch/search/`; look at the results in **`explore.ipynb` section 8**. *Run All*; commit **without outputs**.

| Method | Compares the query with | Returns |
|---|---|---|
| `captions` | each clip's caption text (sentence embeddings, `bge-small`) | clips |
| `siglip_selected` | SigLIP embeddings of the frames stage 4 picked (no captions) | frames → their clip |
| `hybrid` | both rankings merged (reciprocal rank fusion); a clip's SigLIP score = its best frame | clips |
| `siglip_1fps` | **baseline**: SigLIP on 1 frame/s of every video (no gating, no captions) | frames |
| `clip_1fps` | **baseline**: the same with CLIP | frames |

**Test queries** (`queries.csv`, written on the first run, edit or add rows freely): every UCA sentence of the
selected UCA videos (answer = its own time span) and one operator-style sentence per MEVA activity type, e.g.
*"a person opens a car door"* (answers = every labelled instance, any camera).

**Scores**: Recall@1/5/10 = share of queries with a correct moment in the top 1/5/10. "Correct" = temporal IoU ≥ 0.3 or
0.5 with an answer (the returned span is a `moment_s` window around the best frame), or **lenient** = the returned clip
overlaps an answer at all. MRR = 1/rank of the first correct one. **reachable %** = queries whose answer overlaps any
clip at all: the rest were dropped by motion gating, and no search can find them. Two settings: **corpus** (search
every video of the dataset) and **video** (only the query's own video, UCA only, the UCA paper's setting).

| Step | Time on the VM (2 h of video) |
|---|---|
| Build the index (caption embeddings) | seconds; `bge-small` downloads once (~130 MB) |
| Baselines (1 frame/s through SigLIP and CLIP, cached) | a few minutes the first time |
| Evaluate all queries | about a minute |

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

if Path("/mnt/5T_storage").exists():  # team VM: Hugging Face downloads go to the 5T disk
    os.environ.setdefault("HF_HOME", "/mnt/5T_storage/hf_cache")

import search as se

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

## Config

In [2]:
OUT_ROOT = Path(os.environ.get("REWIND_OUT_ROOT", "outputs/batch"))
REMAKE_QUERIES = False     # True = rebuild queries.csv from the ground truth (overwrites your edits)
REBUILD_BASELINES = False  # True = re-embed the 1 frame/s baseline frames even if cached
CFG = {
    "baselines": True,     # False = skip SigLIP / CLIP at 1 frame/s (then only the pipeline's own methods are scored)
    "moment_s": 10.0,      # length of the returned moment around the best frame
    "rrf_k": 60,           # hybrid: rank fusion constant (higher = the two rankings count more equally)
    "top_k": 10,
    "device": "auto",      # "cuda:0" / "cuda:1" if the other group uses one GPU
    "mock": os.environ.get("REWIND_MOCK", "") == "1",   # fake encoders: plumbing check only
}

## 1. Build the index

In [3]:
t0 = time.time()
index = se.SearchIndex(OUT_ROOT, CFG).build(force=REBUILD_BASELINES)
print(f"ready in {time.time() - t0:.0f} s on {index.device}")
display(index.clips.groupby("dataset").agg(videos=("vdir", "nunique"), clips=("clip_id", "size")))

/mnt/5T_storage/clipsExtraction/tools/awsenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4448.54it/s]


[search] 167 clips, 2673 selected frames, 38 videos; text model: BAAI/bge-small-en-v1.5 (sentence model)


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 21350.52it/s]


[search] baselines: 5/38 videos embedded (1504 frames)
[search] baselines: 10/38 videos embedded (3007 frames)
[search] baselines: 15/38 videos embedded (3858 frames)
[search] baselines: 20/38 videos embedded (4432 frames)
[search] baselines: 25/38 videos embedded (4826 frames)
[search] baselines: 30/38 videos embedded (5490 frames)
[search] baselines: 35/38 videos embedded (6765 frames)
[search] baselines: 38/38 videos embedded (7239 frames)
[search] baselines: 7239 frames embedded in 3.2 min
ready in 199 s on cuda


,videos,clips
dataset,,
meva,12,64
uca,26,103


## 2. Test queries with ground truth
Shown: how many per dataset, and a sample. Your own rows can be added to `outputs/batch/search/queries.csv`
(columns `query_id, dataset, source, text, label, video_id, n_gt, gt`; `gt` = JSON list of `[video_id, start_s, end_s]`).

In [4]:
queries = se.load_or_make_queries(index, remake=REMAKE_QUERIES)
display(queries.groupby(["dataset", "source"]).agg(queries=("query_id", "size"), answers=("n_gt", "sum")))
display(queries[queries.dataset == "meva"][["query_id", "label", "text", "n_gt"]])
display(queries[queries.dataset == "uca"][["query_id", "video_id", "text"]].head(10))

,,queries,answers
dataset,source,,
meva,meva_activity,24,680
uca,uca_sentence,194,194


,query_id,label,text,n_gt
0,meva_001,hand_interacts_with_person,a person touches another person with their hand,43
1,meva_002,person_carries_heavy_object,a person carrying a heavy object,6
2,meva_003,person_closes_facility_door,a person closes a door,1
3,meva_004,person_closes_vehicle_door,a person closes a car door,9
4,meva_005,person_embraces_person,two people hugging,2
5,meva_006,person_enters_scene_through_structure,a person comes in through a door,24
6,meva_007,person_enters_vehicle,a person gets into a car,5
7,meva_008,person_exits_scene_through_structure,a person leaves through a door,69
8,meva_009,person_exits_vehicle,a person gets out of a car,3
9,meva_010,person_opens_facility_door,a person opens a door,37


,query_id,video_id,text
24,uca_001,Abuse045_x264,"The woman lying on the sofa kicked the old man sitting next to her, and then..."
25,uca_002,Abuse045_x264,The woman wearing a black suspender stood up from the sofa and pushed the he...
26,uca_003,Abuse045_x264,"The woman stood in front of the old man with her head lowered, seeming to be..."
27,uca_004,Abuse045_x264,The woman pressed the old man on the sofa and shook it vigorously
28,uca_005,Abuse045_x264,"The old man raised his hand, and the woman reached out and hit the old man o..."
29,uca_006,Abuse045_x264,"The woman stood in front of the old man, talked to the old man, pointed, and..."
30,uca_007,Abuse045_x264,"The woman walked to the sofa, took the pillow from the sofa, placed it on th..."
31,uca_008,Abuse045_x264,"The woman walked to the side of the sofa and pushed the old man’s head, then..."
32,uca_009,Abuse045_x264,"The old man wearing a blue vest on the sofa wanted to stand up, but kept sto..."
33,uca_010,Abuse045_x264,The woman stepped forward and slapped the old man on the head hard. Then the...


## 3. Run every query through every method and score it

In [5]:
t0 = time.time()
results, per_query = se.evaluate(index, queries)
met = se.metrics(per_query)
lat = se.latency_probe(index, queries, n=20)
out = se.save(index, queries, results, per_query, met, lat)
print(f"{len(queries)} queries x {per_query.method.nunique()} methods in {time.time() - t0:.0f} s -> {out}")

[search] 50/218 queries
[search] 100/218 queries
[search] 150/218 queries
[search] 200/218 queries
218 queries x 5 methods in 27 s -> outputs/batch/search


## 4. Summary
Main numbers: **R@5 IoU0.3** (strict) and **R@5 lenient** (right clip). Baselines first look at *every* second of
video; the pipeline's methods only see what gating kept, so compare them with **reachable %** in mind.

In [6]:
main = ["queries", "reachable %", "R@1 IoU0.3", "R@5 IoU0.3", "R@10 IoU0.3", "R@5 IoU0.5",
        "R@1 lenient", "R@5 lenient", "R@5 lenient | reachable", "MRR IoU0.3"]
for (ds, setting), g in met.groupby(["dataset", "setting"], sort=False):
    display(Markdown(f"**{ds}, {setting}** ({'all videos of the dataset' if setting == 'corpus' else 'own video only'})"))
    display(g.set_index("method")[main])
display(Markdown("**Time per query** (encode the text + rank everything; ms)"))
display(lat)

**meva, corpus** (all videos of the dataset)

,queries,reachable %,R@1 IoU0.3,R@5 IoU0.3,R@10 IoU0.3,R@5 IoU0.5,R@1 lenient,R@5 lenient,R@5 lenient | reachable,MRR IoU0.3
method,,,,,,,,,,
captions,24,95.83,8.33,25.00,29.17,8.33,33.33,62.50,65.22,0.12
siglip_selected,24,95.83,8.33,16.67,33.33,12.50,16.67,50.00,52.17,0.13
hybrid,24,95.83,4.17,25.00,33.33,8.33,25.00,79.17,82.61,0.13
siglip_1fps,24,95.83,4.17,12.50,29.17,8.33,16.67,33.33,34.78,0.09
clip_1fps,24,95.83,4.17,12.50,20.83,4.17,8.33,29.17,30.43,0.07


**uca, corpus** (all videos of the dataset)

,queries,reachable %,R@1 IoU0.3,R@5 IoU0.3,R@10 IoU0.3,R@5 IoU0.5,R@1 lenient,R@5 lenient,R@5 lenient | reachable,MRR IoU0.3
method,,,,,,,,,,
captions,194,88.66,6.70,14.43,19.59,7.73,10.82,32.47,36.63,0.11
siglip_selected,194,88.66,6.19,17.01,26.29,11.34,13.40,32.99,37.21,0.12
hybrid,194,88.66,7.73,19.59,22.68,12.37,13.92,42.27,47.67,0.12
siglip_1fps,194,88.66,5.67,19.07,30.93,13.40,9.28,27.32,30.23,0.12
clip_1fps,194,88.66,5.15,18.04,26.29,10.82,6.70,22.16,20.93,0.11


**uca, video** (own video only)

,queries,reachable %,R@1 IoU0.3,R@5 IoU0.3,R@10 IoU0.3,R@5 IoU0.5,R@1 lenient,R@5 lenient,R@5 lenient | reachable,MRR IoU0.3
method,,,,,,,,,,
captions,194,88.66,12.37,35.05,38.66,20.10,32.99,82.99,93.60,0.21
siglip_selected,194,88.66,17.53,40.72,52.06,23.71,39.69,78.35,86.63,0.27
hybrid,194,88.66,14.95,35.05,38.66,19.07,37.11,82.47,93.02,0.23
siglip_1fps,194,88.66,17.01,44.85,62.37,25.77,27.32,66.49,69.19,0.29
clip_1fps,194,88.66,15.46,47.94,61.86,28.35,21.65,69.07,70.93,0.29


**Time per query** (encode the text + rank everything; ms)

,median,p95
method,,
captions,36.6,39.7
siglip_selected,37.4,38.7
hybrid,35.7,37.4
siglip_1fps,40.7,63.4
clip_1fps,37.8,40.8


## 5. Try a query
Any sentence. `method`: `hybrid` (default), `captions`, `siglip_selected`, `siglip_1fps`, `clip_1fps`. Filters:
`dataset="meva"|"uca"`, `video_id=…`, `camera_id=…`. The frames are in `explore.ipynb` section 8.

In [7]:
QUERY = "a person gets out of a car"
hits = index.search(QUERY, k=5, method="hybrid")
display(hits[["rank", "dataset", "video_id", "camera_id", "clip_start", "clip_end", "best_t", "score", "summary"]])

,rank,dataset,video_id,camera_id,clip_start,clip_end,best_t,score,summary
0,1,uca,Shooting052_x264,Shooting052_x264,50.2,81.4,65.13,0.031754,"A person walks past two parked cars in a parking lot, then exits the frame."
1,2,uca,Stealing109_x264,Stealing109_x264,133.6,166.7,160.27,0.031010,"A person walks towards a silver SUV, opens the rear door, and appears to ret..."
2,3,uca,Stealing109_x264,Stealing109_x264,201.8,220.8,208.47,0.030303,"A person wearing dark clothing approaches a silver SUV, opens the rear door,..."
3,4,uca,Shooting047_x264,Shooting047_x264,131.2,163.4,162.37,0.030092,"A person walks towards a parked car, opens its door, and exits. Another indi..."
4,5,uca,Shooting047_x264,Shooting047_x264,163.4,195.6,169.30,0.030018,"A person walks towards a parked car, opens its door, and gets inside. Anothe..."


## 6. Operator-style queries (no ground truth)
Free-text questions an operator might ask (objective 5): event, person activity, and time conditions. No score: look at
the answers (`explore.ipynb` section 8 shows their frames). Saved to `search/demo_results.csv`.

In [8]:
DEMO_QUERIES = [
    "someone running", "a fight between two people", "a person falls to the ground", "a car crash",
    "a person carrying a bag", "a group of people standing together", "a person on a bicycle",
    "a car parked by the door", "a person in a red shirt", "someone walking at night",
    "a person breaking into a car", "people walking into a building", "a dog", "a person sitting on a bench",
    "smoke or fire", "a truck", "a person using a phone", "two people shaking hands", "a person stealing", "an empty room",
]
demo = pd.concat([index.search(q, k=3, method="hybrid").assign(query=q) for q in DEMO_QUERIES], ignore_index=True)
demo.to_csv(OUT_ROOT / "search" / "demo_results.csv", index=False)
display(demo[demo["rank"] == 1][["query", "dataset", "video_id", "clip_start", "clip_end", "summary"]])

,query,dataset,video_id,clip_start,clip_end,summary
0,someone running,uca,Vandalism049_x264,203.2,255.800,"A person wearing a dark shirt and pants walks towards a gate, opens it, and ..."
3,a fight between two people,uca,Fighting049_x264,4.8,38.600,A man in a black jacket stands up and appears to be involved in a physical a...
6,a person falls to the ground,uca,Fighting045_x264,80.6,134.800,A group of people gather around an individual lying on the ground. The crowd...
9,a car crash,uca,Vandalism044_x264,2.8,40.900,"A person walks between two parked cars, then another person appears near a s..."
12,a person carrying a bag,uca,Abuse045_x264,9.6,23.200,"A person walks into the room carrying a child, then walks away."
15,a group of people standing together,meva,2018-03-13.15-55-00.16-00-00.school.G421.r13,144.6,181.200,"A group of people stands in a room, some move slightly while others remain s..."
18,a person on a bicycle,uca,Fighting045_x264,80.6,134.800,A group of people gather around an individual lying on the ground. The crowd...
21,a car parked by the door,uca,Stealing109_x264,252.4,299.000,A silver SUV is parked near a building. A person wearing dark clothing walks...
24,a person in a red shirt,uca,Shooting052_x264,111.6,138.400,A person walks past the red car in the parking lot.
27,someone walking at night,uca,Vandalism049_x264,147.2,156.600,"A person walks down the sidewalk, followed by two more individuals. The grou..."
